# Trend against the signal gate

Score the trend lane against the signal gate. Two kinds of result are scored.

- The primary test is the always-on lookback sweep, the per coin and market books on \$10M coins. The rule and the grid were named before the wide data was run. The 4-week lookback is the grid's winner, so it is priced against the whole grid.
- The moderate-month sleeve is exploratory. It was found by searching a map of conditions over the full sample, so it can at best park.

Every trial the lane has run counts toward the luck bar. The trials on the wide data are rebuilt here as return series, so their overlap is measured rather than assumed. About 30 trials on the old ten-coin sample have no series on this data and are counted as fully independent, which overstates the penalty.

Every book is costed off the measured ladder at a \$100k research book.

## Setup

Load the shared book, the gate and the cost ladder. Build the sweep, the sleeve and the ledger of every trial.

In [1]:
import warnings
import numpy as np
import pandas as pd
from qp_research import condition, costs, gate, panel
from qp_research.book import Book, inverse_vol, gated as gated_weights
import signals
from signals import sharpe

warnings.filterwarnings('ignore')
pd.set_option('display.width', 200)
pd.set_option('display.max_colwidth', 80)

p = panel.load()
W, LIQ, VOL, F = p.returns, p.liquidity, p.vol, p.funding
fit = costs.fit(costs.sample(['2023-03-10', '2024-03-10', '2025-03-10', '2026-06-10'], p.liquidity))
SIZE, OLD_TRIALS = 1e5, 30
tb = Book(p, costs.Taker(costs.FEES['usdm'], fit, SIZE))
LOOKBACKS = [1, 2, 3, 4, 5, 6, 8, 12]

def run(w, **kw):
    return tb.run(w, **kw)

def btc(L):
    w = pd.DataFrame(0.0, index=W.index, columns=W.columns)
    w['BTCUSDT'] = np.sign(W['BTCUSDT'].rolling(L, min_periods=L).sum()).shift(1)
    return w

sweep = {}
for L in LOOKBACKS:
    sweep[f'per coin $1M L{L}'] = signals.per_coin(p, L, 1e6)
    sweep[f'per coin $10M L{L}'] = signals.per_coin(p, L)
    sweep[f'market $10M L{L}'] = signals.market(p, L)
    sweep[f'BTC L{L}'] = btc(L)
nets = {k: run(w).net for k, w in sweep.items()}

per_coin_w, market_w = signals.per_coin(p, 4), signals.market(p, 4)
idx = run(per_coin_w).index
cells = condition.cells(condition.market_conditions(p), idx)

def moderate(n_ret=4, n_vol=12, q=1 / 3):
    return (condition.past_cut(condition.strength(p, n_ret, n_vol), q) == 1).reindex(idx).fillna(False)

on = moderate()
sleeve = tb.sleeve(per_coin_w, on)
long_b, short_b = run(signals.basket(p)).net, run(-signals.basket(p)).net
print(f'weeks {len(idx)}   sleeve weeks {int(on.sum())}')

weeks 336   sleeve weeks 98


## The trial ledger

Rebuild every configuration the lane tried on the wide data as a weekly return series. The sweep, the residual books from the decomposition, the funding-aware sign, the breakout and the cleanliness thirds, the small coins, every cell of the condition map in both books, the open interest thirds, and every robustness variant of the moderate month. A conditional trial is its gated book, flat when its condition is off.

Then count them in effective trials, the number of independent tests with the same expected best, so near copies count as one.

In [2]:
def signal_book(sig, floor=1e7, hi=np.inf):
    return run(inverse_vol(p, sig.shift(1), floor, hi)).net

def gated(w, on_mask):
    return run(gated_weights(w, on_mask)).net

mret = condition.market_return(p)
level = W.cumsum()
ledger = dict(nets)
for L in [1, 2, 3, 4, 6, 8, 12]:
    resid = W.rolling(L, min_periods=L).sum().sub(mret.rolling(L, min_periods=L).sum(), axis=0)
    ledger[f'residual L{L}'] = signal_book(np.sign(resid))
    if L == 4:
        ledger['residual reversal L4'] = signal_book(-np.sign(resid))
for L in [1, 2, 4, 8, 12]:
    ret = W.rolling(L, min_periods=L).sum()
    ledger[f'funding-aware L{L}'] = signal_book(np.sign(ret - F.rolling(L, min_periods=L).sum()))
    brk = pd.DataFrame(0.0, index=W.index, columns=W.columns) \
        .mask(level >= level.rolling(L, min_periods=L).max(), 1.0).mask(level <= level.rolling(L, min_periods=L).min(), -1.0)
    ledger[f'breakout L{L}'] = signal_book(brk)
    ledger[f'small coins L{L}'] = signal_book(np.sign(ret), 1e6, 1e7)
    if L > 1:
        er = (ret.abs() / W.abs().rolling(L, min_periods=L).sum()).where((LIQ >= 1e7) & VOL.notna() & ret.notna())
        lo, hi = er.quantile(1 / 3, axis=1), er.quantile(2 / 3, axis=1)
        for name, m in [('low', er.le(lo, axis=0)), ('mid', er.gt(lo, axis=0) & er.lt(hi, axis=0)), ('high', er.ge(hi, axis=0))]:
            ledger[f'cleanliness {name} L{L}'] = signal_book(np.sign(ret).where(m))
for n, m in cells.items():
    ledger[f'per coin, {n}'] = gated(per_coin_w, m)
    ledger[f'market, {n}'] = gated(market_w, m)

majors = ['BTCUSDT', 'ETHUSDT', 'SOLUSDT', 'BNBUSDT', 'XRPUSDT', 'DOGEUSDT']
oi = signals.open_interest(majors, '2020-09-01', '2026-09-01')
contracts = np.log(oi.resample('W-SUN').last().reindex(W.index)).sub(W[majors].cumsum(), axis=0)
for k, v in {'OI growth': contracts.diff(4).mean(axis=1).shift(1),
             'OI level': (contracts - contracts.rolling(26, min_periods=13).mean()).mean(axis=1).shift(1)}.items():
    lab = condition.thirds(v).reindex(idx)
    for t, name in enumerate(['low', 'mid', 'high']):
        ledger[f'per coin, {k} {name}'] = gated(per_coin_w, lab == t)

for q in [0.25, 0.5]:
    ledger[f'moderate, cut {q:.0%}'] = gated(per_coin_w, moderate(q=q))
for nr in [2, 4, 8]:
    for nv in [8, 12, 26]:
        if (nr, nv) != (4, 12):
            ledger[f'moderate, {nr}w move {nv}w vol'] = gated(per_coin_w, moderate(nr, nv))
for L in LOOKBACKS:
    if L != 4:
        ledger[f'moderate, per coin L{L}'] = gated(signals.per_coin(p, L), on)
ledger['moderate, market L4'] = gated(market_w, on)
ledger = pd.DataFrame(ledger)

primary = pd.DataFrame(nets)
neff_primary = gate.effective_trials(primary, extra=OLD_TRIALS)
neff_lane = gate.effective_trials(ledger, extra=OLD_TRIALS)
print(f'primary family   {primary.shape[1]} sweep cells and {OLD_TRIALS} old trials   effective trials {neff_primary:.1f}')
print(f'whole lane       {ledger.shape[1]} trials and {OLD_TRIALS} old trials       effective trials {neff_lane:.1f}')

primary family   32 sweep cells and 30 old trials   effective trials 42.1
whole lane       125 trials and 30 old trials       effective trials 73.6


The lane has run 125 configurations on the wide data and about 30 on the old sample. They overlap heavily, so they are worth about 74 independent tests. The primary family, the 32 sweep cells and the old trials, is worth about 42.

That sets the luck bar for the always-on books at a Sharpe of 0.86 over the full span, the best a search this size reaches with no edge at all. For the sleeve, priced against the whole lane over its 98 weeks, the bar is 1.75.

## Data and cost checks

Two checks read the data rather than a book. How many halt gaps the book held through and what they booked, and every halt gap beyond 50% either way, so a redenomination missing from the named list shows up. Then the fee charged per leg.

In [3]:
w = tb.hold(per_coin_w)
gaps = tb.p.gaps.reindex_like(w)
held = w.ne(0) & gaps.notna()
print(f'halt gaps held through {int(held.sum().sum())}   net contribution {(w * np.expm1(gaps.fillna(0.0))).sum().sum():+.4f}')
big = tb.p.gaps.stack()
big = big[big.abs() > np.log(1.5)]
print(f'halt gaps beyond 50% either way {len(big)}')
print(np.expm1(big).round(2).sort_values().to_string())
print(f"fee per leg {costs.FEES['usdm']} bps")

halt gaps held through 4   net contribution -0.0003
halt gaps beyond 50% either way 8
day                        symbol   
2025-07-13 00:00:00+00:00  PUMPUSDT    -0.89
2025-03-30 00:00:00+00:00  MAVIAUSDT   -0.69
2025-05-04 00:00:00+00:00  CTKUSDT     -0.55
2023-04-02 00:00:00+00:00  TLMUSDT     -0.43
2025-07-27 00:00:00+00:00  SLPUSDT     -0.35
                           CVXUSDT      1.00
2025-04-20 00:00:00+00:00  AERGOUSDT    1.06
2025-12-28 00:00:00+00:00  LITUSDT      5.13
fee per leg 5.0 bps


The book held through 4 halt gaps and they cost it 0.03% in total, so the halt fix changes no result.

Eight halt gaps moved past 50% up or past a third down, measured in log terms. The largest are LIT up fivefold in December 2025 and PUMP down 89% in July 2025. None is on the named list of redenominations, so each is booked as a real move. Whether any of them is a redenomination is not settled here, and with 4 held gaps worth 0.03% it moves nothing.

The fee is 5 bps a leg, the current USD-M taker rate.

## Scoring

Each result is run through every check the data can settle. The rest are stated with the reason they pass or stay open.

- S1 prices the primary books against the primary family and the sleeve against the whole lane.
- S3 trades the lookback chosen inside each walk-forward fold for the primary books, and the condition chosen inside each fold for the sleeve. In sample is the same book over the history before the first test block.
- S4 reads the lookbacks either side for the primary books. For the sleeve it moves each setting of the condition by about 20% either way, the cut point, the move window and the volatility window, and the lookback one step either side.
- S5 is the probability the choice overfit, over the lookback grid for the primary books and over the condition map for the sleeve.
- S6 is lift over the long and the short basket and over persistence, the one-week lookback.
- T1 reruns each result with its signal one week older. It must do no better.

In [4]:
def check(ok, note):
    return ('pass' if ok else 'fail', note)

def primary_score(name):
    grid = pd.DataFrame({L: nets[f'{name} L{L}'] for L in LOOKBACKS}).dropna()
    x = nets[f'{name} L4']
    prob, bar = gate.significance(x, neff_primary)
    oos, picked = gate.walk_forward(grid, folds=6, min_train=104)
    s3, z3 = gate.oos_agree(x.loc[:oos.index[0]].iloc[:-1], oos)
    s4, z4 = gate.neighbours_agree(grid[4], [grid[3], grid[5]])
    s2, keep = gate.year_stability(x)
    p5 = gate.pbo(grid)
    base = {'long basket': sharpe(long_b), 'short basket': sharpe(short_b), 'persistence': sharpe(grid[1])}
    late = sharpe(run(sweep[f'{name} L4'].shift(1)).net)
    twice = sharpe(run(sweep[f'{name} L4'], cost_mult=2.0).net)
    boot = gate.bootstrap(x)
    return x, prob, {
        'S1': (('pass' if prob >= gate.CONFIDENCE else 'park' if prob >= gate.PARK else 'fail'),
               f'probability {prob:.2f}, Sharpe {sharpe(x):.2f} against a luck bar of {bar:.2f}'),
        'S2': check(s2, f'worst year removed keeps {keep:.0%} of the Sharpe'),
        'S3': check(s3, f'walk forward {sharpe(oos):.2f} against {sharpe(x.loc[:oos.index[0]].iloc[:-1]):.2f} in sample, drop z {z3:.1f}, picked {[int(p) for p in picked]}'),
        'S4': check(s4, f'L3 {sharpe(grid[3]):.2f}, L4 {sharpe(grid[4]):.2f}, L5 {sharpe(grid[5]):.2f}, gap z {z4[0]:.1f} and {z4[1]:.1f}'),
        'S5': check(p5 < 0.5, f'probability of overfit {p5:.2f}'),
        'S6': check(sharpe(x) > max(base.values()), ', '.join(f'{k} {v:.2f}' for k, v in base.items())),
        'T1': check(late <= sharpe(x), f'one week older {late:.2f}'),
        'C4': check(twice > 0, f'twice cost {twice:.2f}'),
        'V1': check(boot['sharpe 5%'] > 0, f"5th percentile {boot['sharpe 5%']:.2f}, ends above zero {boot['ends above zero']:.0%}, median drawdown {boot['drawdown 50%']:.2f}")}

def sleeve_score():
    x = sleeve
    prob, bar = gate.significance(x, neff_lane)
    oos, on_oos, picked = condition.walk_forward(tb, per_coin_w, cells, folds=4, history=156, min_on=20)
    s3, z3 = gate.oos_agree(x.loc[:oos.index[0]].iloc[:-1], oos[on_oos])
    variants = {'cut 27%': moderate(q=0.267), 'cut 40%': moderate(q=0.4), 'move 3w': moderate(n_ret=3), 'move 5w': moderate(n_ret=5),
                'vol 10w': moderate(n_vol=10), 'vol 14w': moderate(n_vol=14)}
    near = {k: gated(per_coin_w, m).reindex(idx) for k, m in variants.items()}
    near.update({'L3': gated(signals.per_coin(p, 3), on).reindex(idx), 'L5': gated(signals.per_coin(p, 5), on).reindex(idx)})
    s4, z4 = gate.neighbours_agree(gated(per_coin_w, on).reindex(idx), list(near.values()))
    s2, keep = gate.year_stability(x)
    p5 = gate.pbo(ledger[[c for c in ledger if c.startswith('per coin, ') and 'OI' not in c]])
    base = {'long basket': sharpe(long_b.reindex(idx)[on]), 'short basket': sharpe(short_b.reindex(idx)[on]),
            'persistence': sharpe(tb.sleeve(signals.per_coin(p, 1), on))}
    late = sharpe(tb.sleeve(per_coin_w.shift(1), on))
    twice = sharpe(tb.sleeve(per_coin_w, on, cost_mult=2.0))
    boot = gate.bootstrap(x)
    return x, prob, {
        'S1': (('pass' if prob >= gate.CONFIDENCE else 'park' if prob >= gate.PARK else 'fail'),
               f'probability {prob:.2f}, Sharpe {sharpe(x):.2f} against a luck bar of {bar:.2f}, {len(x)} weeks'),
        'S2': check(s2, f'worst year removed keeps {keep:.0%} of the Sharpe'),
        'S3': check(s3, f'condition chosen in fold {sharpe(oos[on_oos]):.2f} against {sharpe(x.loc[:oos.index[0]].iloc[:-1]):.2f} before, drop z {z3:.1f}, picked {list(picked.values())}'),
        'S4': check(s4, ', '.join(f'{k} gap z {z:.1f}' for k, z in zip(near, z4))),
        'S5': check(p5 < 0.5, f'probability of overfit across the map {p5:.2f}'),
        'S6': check(sharpe(x) > max(base.values()), ', '.join(f'{k} {v:.2f}' for k, v in base.items())),
        'T1': check(late <= sharpe(x), f'one week older {late:.2f}'),
        'C4': check(twice > 0, f'twice cost {twice:.2f}'),
        'V1': check(boot['sharpe 5%'] > 0, f"5th percentile {boot['sharpe 5%']:.2f}, ends above zero {boot['ends above zero']:.0%}, median drawdown {boot['drawdown 50%']:.2f}")}

STATED = {
    'D1': ('pass', 'January 2020 to September 2026, the 2021 bull, the 2022 bear and the 2023 to 2024 recovery'),
    'D2': ('pass', 'redenominations from a named list, halt gaps beyond 50% printed above for review'),
    'D3': ('pass', 'every USDT perp the archive has carried, delisted included'),
    'D4': ('pass', 'daily bars keyed on close time, funding on settlement stamps'),
    'D5': ('pass', 'returns never span a halt in a signal, positions hold through halts and book the gap'),
    'T2': ('pass', 'lookback and condition choices scored in fold under S3'),
    'T3': ('pass', 'trailing features, cut points from earlier weeks, one bar for every coin'),
    'T4': ('pass', 'no fitted model'),
    'T5': ('open', 'universe and liquidity as of the bar, but the cost ladder is fit on 2023 to 2026 depth and applied from 2020. Closes with depth from 2020 to 2022'),
    'S7': ('pass', 'every Sharpe is on one weekly book return, one row per week'),
    'C1': ('pass', f"{costs.FEES['usdm']} bps taker per leg, the current USD-M rate"),
    'C2': ('pass', 'measured ladder at a $100k book'),
    'C3': ('pass', 'funding paid and received every week on every open leg'),
    'V2': ('pass', 'the path through the breaks is read in the bounding notebook')}

results = {'per coin $10M, always on': primary_score('per coin $10M'),
           'market $10M, always on': primary_score('market $10M'),
           'moderate month sleeve': sleeve_score()}
order = ['D1', 'D2', 'D3', 'D4', 'D5', 'T1', 'T2', 'T3', 'T4', 'T5', 'S1', 'S2', 'S3', 'S4', 'S5', 'S6', 'S7', 'C1', 'C2', 'C3', 'C4', 'V1', 'V2']
card = pd.DataFrame({name: {k: (sc | STATED).get(k, ('', ''))[0] for k in order} for name, (_, _, sc) in results.items()})
print(card.to_string())
print()
for name, (x, prob, sc) in results.items():
    scored = sc | STATED
    failed = [k for k in order if scored[k][0] == 'fail' and k != 'S1']
    opened = [k for k in order if scored[k][0] == 'open']
    trials = neff_lane if 'sleeve' in name else neff_primary
    print(f'{name}   {gate.stamp(failed, prob, sharpe(x), trials, 52, open_items=opened, exploratory="sleeve" in name)}')
    for k in order:
        if k in sc:
            print(f'    {k}  {sc[k][0]:5} {sc[k][1]}')
    print()

   per coin $10M, always on market $10M, always on moderate month sleeve
D1                     pass                   pass                  pass
D2                     pass                   pass                  pass
D3                     pass                   pass                  pass
D4                     pass                   pass                  pass
D5                     pass                   pass                  pass
T1                     pass                   pass                  pass
T2                     pass                   pass                  pass
T3                     pass                   pass                  pass
T4                     pass                   pass                  pass
T5                     open                   open                  open
S1                     fail                   fail                  fail
S2                     pass                   pass                  pass
S3                     pass                   pass 

Neither always-on book passes. Both fail on significance and on the lookback neighbours.

- The per coin book makes 0.85 against a luck bar of 0.86, real with probability 0.49. The market book makes 0.79, real with probability 0.43. Both sit at the bar a search of this size reaches by chance.
- Over the weeks all lookbacks share, the per coin book makes 0.88 at 4 weeks, 0.79 at 5 and 0.17 at 3. The gap to 3 weeks is 2.7 standard errors, far wider than two books this alike should show. The market book makes 0.82, 0.46 and 0.01.

Every other check passes. The walk forward picks 4 weeks at every fold and holds within chance of in sample, 0.71 against 1.09 per coin and 0.72 against 0.94 for the market. The probability the lookback choice overfit is 0.29 and 0.40. Removing any year keeps at least 67% of the Sharpe. Both beat the long basket at 0.58, the short basket and persistence. They survive twice the cost, and the bootstrap floor is 0.36 and 0.29. Both stamp `gate fail S1 S4, open T5`.

The sleeve fails on more counts. Its 1.54 over 98 weeks sits under a luck bar of 1.75, real with probability 0.38. The map it came from is 0.67 likely to have picked a lucky cell. And inside its own weeks simpler books do as well. Persistence makes 1.70 on the same condition and the short basket 0.90. It passes the neighbour check, every nudge to the cut, the windows and the lookback agrees with it, and removing any year keeps 84% of its Sharpe. It stamps `gate fail S1 S5 S6, open T5`.

## Verdict

Trend on Binance perps does not pass the signal gate.

The always-on 4-week book earns. It makes 0.85 net per coin and 0.80 for the market book over six and a half years, holds in walk forward, survives cost at a \$100k book, and no year carries it. But it clears the luck bar of the search that found it with probability under a half, and 3 weeks earns almost nothing next to 4 and 5, so the 4-week setting cannot be told apart from a lucky cell.

The moderate-month sleeve fails on search, overfitting and baseline. It was a hindsight pick from a map, and inside its own weeks the one-week rule and the short basket do as well.

What remains is a market-timing premium near 0.8 whose evidence falls short of the bar, and a condition search that paid out of sample when it was run in fold. Neither is a pass.